In [46]:
from scipy.stats import norm, t
import numpy as np

# Sample size for A/B tests
Sample size required for an A/B test is dependent on
- continuous vs rate evaluation metric
- one-side hypothesis (is treatment better than control) vs two-side hypothesis (is treatment different, either better or worse from control)
- confidence threshold (critical p-value for rejecting the null hypothesis)
- Power of the test (probability of detecting an effect if it exists)
- traffic allocation ratio between experiment and control
- desired minimum detectable effect size

## Sample size for continuous evaluation metric, e.g. revenue per visit

### Formula
The formula for sample size required under a one-sided hypothesis is given by

\begin{align*}
\mathbf{N} &= (r+1) \sigma^2 \Bigl(\frac{q_\alpha + q_{1-\beta}}{\Delta\mu} \Bigr)^2 \\
\\
\text{Where}& \\
\mathbf{N} &= \text{the sample size required for the treatment group} \\
r &= \frac{N_{treatment}}{N_{control}} \text{ traffic allocation ratio, between (0,1) } \\
\sigma^2 &= \text{estimate of the population variance of the evaluation metric} \\
\alpha &= \text{confidence level threshold or critical p-value, typically 5%} \\
\beta &= \text{power of the test, typically 80%} \\
q_x &= \text{x level quantile of the standard normal distribution} \\
\Delta\mu &= \text{desired miniumum detectable difference in the evaluation metric between treatment and control} \\
\\
\text{For common}&\text{ usecases of 5% critical p-value, 80% power, and equal traffic allocation, this simplifies to }\\
\\
\mathbf{N} &\approx 12.4 \frac{\sigma^2}{\Delta\mu^2} \\
\\
\text{Formula for}&\text{ a two-sided hypothesis is very similar with a small modification on }\alpha\\
\\
\mathbf{N} &= (r+1) \sigma^2 \Bigl(\frac{q_{\alpha/2} + q_{1-\beta}}{\Delta\mu} \Bigr)^2 \\
\end{align*}


In [47]:
# one sided: treatment better than control
# returns the treatment sample size
def sample_size_mean_1side(sig, power, pop_var, lift, allocation_ratio=1):
    q_alpha = norm.isf(sig)
    q_beta = norm.isf(1-power)
    n = (allocation_ratio + 1) * (q_alpha + q_beta) ** 2 * pop_var / lift ** 2
    return int(np.ceil(n))

#two-sided: both better and worse
# returns the treatment sample size
def sample_size_mean_2side(sig, power, pop_var, lift, allocation_ratio=1):
    return sample_size_mean_1side(sig/2, power, pop_var, lift, allocation_ratio)

### Example usage
We want to run an experiment to measure the effect of a new hero image on revenue per visit. We know from historical data, revenue per visit is 10 dollars with a variance of 100, and we want to be able to detect an increase of at least 5% (or 0.5 dollars) with a confidence threshold of 5% and power of 80%, what is the sample size required for this experiment?

In [48]:
sample_size_mean_1side(0.05, 0.8, 100, 0.5, allocation_ratio=1)

4947

In [49]:
sample_size_mean_2side(0.05, 0.8, 100, 0.5, allocation_ratio=1)

6280

## Sample size for rate evaluation metric, e.g. click through rate

### Formula
The formula for sample size required under a one-sided hypothesis is given by

\begin{align*}
\mathbf{N} &= \Bigl[r(p+\Delta p)[1-(p+\Delta p)] + p(1-p)\Bigr] \Bigl(\frac{q_\alpha + q_{1-\beta}}{\Delta p} \Bigr)^2 \\
\\
\text{Where}& \\
\mathbf{N} &= \text{the sample size required for the treatment group} \\
r &= \frac{N_{treatment}}{N_{control}} \text{ traffic allocation ratio, between (0,1) } \\
p &= \text{estimate of the baseline rate in the control population} \\
\Delta p &= \text{desired miniumum detectable difference in the evaluation metric between treatment and control} \\
\alpha &= \text{confidence level threshold or critical p-value, typically 5%} \\
\beta &= \text{power of the test, typically 80%} \\
q_x &= \text{x level quantile of the standard normal distribution} \\
\\
\text{For common}&\text{ usecases of 5% critical p-value, 80% power, and equal traffic allocation, this simplifies to }\\
\\
\mathbf{N} &\approx 12.4 \frac{p(1-p)}{\Delta p^2} \\
\\
\text{Formula for}&\text{ a two-sided hypothesis is very similar with a small modification on }\alpha\\
\\
\mathbf{N} &= \Bigl[r(p+\Delta p)[1-(p+\Delta p)] + p(1-p)\Bigr] \Bigl(\frac{q_{\alpha/2} + q_{1-\beta}}{\Delta p} \Bigr)^2 \\
\end{align*}


In [50]:
# one sided - treatment better than control
# returns the treatment sample size
def sample_size_proportion_1side(sig, power, base_rate, abs_improvement, allocation_ratio=1):
    z_alpha = norm.isf(sig)
    z_beta = norm.isf(1-power)
    improve_rate = base_rate + abs_improvement
    n = (z_alpha + z_beta) ** 2 * (allocation_ratio*improve_rate*(1-improve_rate) + base_rate*(1-base_rate)) / (abs_improvement) ** 2
    return int(np.ceil(n))


#both better and worse
# returns the treatment sample size
def sample_size_proportion_2side(sig, power, base_rate, abs_improvement, allocation_ratio=1):
    return sample_size_proportion_1side(sig/2, power, base_rate, abs_improvement, allocation_ratio)

### Example usage
We want to run an experiment to measure the effect of a new hero image on sign up rate. We know from historical data, sign up rate is about 10%, and we want to be able to detect an increase of at least 5% (0.5% in absolute terms) with a confidence threshold of 5% and power of 80%, what is the sample size required for this experiment?

In [51]:
sample_size_proportion_1side(0.05, 0.8, 0.1, 0.005, allocation_ratio=1)

45498

In [52]:
sample_size_proportion_2side(0.05, 0.8, 0.1, 0.005, allocation_ratio=1)

57760

# Analysis procedure for A/B tests

## Analysis for continuous evaluation metric, e.g. revenue per visit

Use the t-test for comparing continuous evaluation metrics across groups, assuming one-sided hypothesis

\begin{align*}
S_p^2 &= \frac{S_1^2}{N_1} + \frac{S_2^2}{N_2} \\
t &= \frac{\Delta\mu}{\sqrt{S_p^2}} \\
df &= \frac{(S_p^2)^2}{\frac{\frac{S_1^2}{N_1}}{N_1-1} + \frac{\frac{S_2^2}{N_2}}{N_2-1}}\\
p-value &= 1-cdf(t, df) \\
\\
\text{Reject null} &\text{ hypothesis if the p-value is below the desired confidence level, typically 5%}
\\
\\
\text{Where}& \\
S_x^2 &= \text{Sample variance of the treatment/control groups} \\
N_x &= \text{Sample size of the treatment/control groups} \\
\Delta\mu &= \text{evaluation metric of the treatment group - evaluation metric of the control group} \\
df &= \text{degrees of freedom of the experiment} \\
cdf &= \text{cumulative distribution function of the t-distribution}
\\
\\
\text{Formula for}&\text{ a two-sided hypothesis is very similar with a small modification on p-value}\\
\\
p-value &= 2(1-cdf(|t|, df)) \\
\\
\text{If the confidence}&\text{ interval around the difference of evaluation metric between treatment and control is of interest}\\
\\
ci &= \bigl(\Delta\mu - q_{(1-conf)/2}\sqrt{S_p^2}, \enspace \Delta\mu + q_{(1-conf)/2}\sqrt{S_p^2}) \\
\\
\text{Where}& \\
conf &= \text{the confidence level of the confidence interval, typically 95%} \\
q_x &= \text{x level quantile of the standard normal distribution} \\
\end{align*}


In [53]:
# mu1 > mu2
# does not assume equal variance between the 2 samples
def mean_test_1side(var_treat, var_control, mu_treat, mu_control, n_treat, n_control):
    diff = mu_treat - mu_control
    s = (var_treat/n_treat + var_control/n_control) ** 0.5
    t_stat = diff / s
    df = (var_treat/n_treat + var_control/n_control) ** 2 / ((var_treat/n_treat) ** 2 / (n_treat-1) + (var_control/n_control) ** 2 / (n_control-1))
    prob = 1 - t.cdf(t_stat, df)
    return prob


# mu1 != mu2
# does not assume equal variance between the 2 samples
def mean_test_2side(var_treat, var_control, mu_treat, mu_control, n_treat, n_control):
    diff = mu_treat - mu_control
    s = (var_treat/n_treat + var_control/n_control) ** 0.5
    t_stat = diff / s
    df = (var_treat/n_treat + var_control/n_control) ** 2 / ((var_treat/n_treat) ** 2 / (n_treat-1) + (var_control/n_control) ** 2 / (n_control-1))
    prob = 1 - t.cdf(abs(t_stat), df)
    return 2*prob

# confidence level
def mean_test_ci(var_treat, var_control, mu_treat, mu_control, n_treat, n_control, conf=0.95):
    diff = mu_treat - mu_control
    s = (var_treat/n_treat + var_control/n_control) ** 0.5
    df = (var_treat/n_treat + var_control/n_control) ** 2 / ((var_treat/n_treat) ** 2 / (n_treat-1) + (var_control/n_control) ** 2 / (n_control-1))
    t_crit = t.isf((1-conf)/2, df)
    return (diff - t_crit*s, diff + t_crit*s)

### Example usage
We ran an experiment to measure the effect of a new hero image on revenue per visit. For the control group we saw revenue per visit of 10 dollars with a variance of 100 across 5500 visits, and for the treatment group we saw revenue per visit of 10.5 dollars with a variance of 110 across 5000 visits, can we reject the null hypothesis at the confidence level of 5%? And what's the confidence level around the improvement?

In [57]:
print('p-value of the one-sided test is {:0.1%}'.format(mean_test_1side(110, 100, 10.5, 10, 5000, 5500)))

p-value of the one-side test is 0.6%


In [58]:
print('p-value of the two-sided test is {:0.1%}'.format(mean_test_2side(110, 100, 10.5, 10, 5000, 5500)))

p-value of the two-side test is 1.3%


In [62]:
print('The 95% confidence interval of the improvement is between ({:0.1f} dollars, {:0.1f} dollars)'.format(*mean_test_ci(110, 100, 10.5, 10, 5000, 5500)))

The 95% confidence interval of improve is between (0.1 dollars, 0.9 dollars)


#### Treatment beats control at a confidence level of 5% (0.6% < 5%)

## Analysis for rate evaluation metric, e.g. click through rate

Use the z-test for comparing rate evaluation metrics across groups, assuming one-sided hypothesis

\begin{align*}
p_{pool} &= \frac{p_1N_1 + p_2N_2}{N_1 + N_2} \\
SE &= \sqrt{p_{pool}(1-p_{pool})(\frac{1}{N_1} + \frac{1}{N_2})} \\
z &= \frac{\Delta p}{SE} \\
p-value &= 1-cdf(z) \\
\\
\text{Reject null} &\text{ hypothesis if the p-value is below the desired confidence level, typically 5%}
\\
\\
\text{Where}& \\
p_x &= \text{evaluation metric rate of the treatment/control groups} \\
N_x &= \text{Sample size of the treatment/control groups} \\
\Delta p &= \text{evaluation metric of the treatment group - evaluation metric of the control group} \\
cdf &= \text{cumulative distribution function of the standard normal distribution}
\\
\\
\text{Formula for}&\text{ a two-sided hypothesis is very similar with a small modification on p-value}\\
\\
p-value &= 2(1-cdf(|z|)) \\
\\
\text{If the confidence}&\text{ interval around the difference of evaluation metric between treatment and control is of interest}\\
\\
ci &= \bigl(\Delta p - q_{(1-conf)/2}SE, \enspace \Delta p + q_{(1-conf)/2}SE \bigr) \\
\\
\text{Where}& \\
conf &= \text{the confidence level of the confidence interval, typically 95%} \\
q_x &= \text{x level quantile of the standard normal distribution} \\
\end{align*}


In [39]:
# p1 > p2
def proportion_test_1side(p_treat, p_control, n_treat, n_control):
    pooled_p = (p_treat*n_treat+p_control*n_control) / (n_treat+n_control)
    standard_error = (pooled_p * (1 - pooled_p) * (1/n_treat + 1/n_control)) ** 0.5
    z = (p_treat - p_control) / standard_error
    prob = 1 - norm.cdf(z)
    return prob

# p1 != p2
def proportion_test_2side(p_treat, p_control, n_treat, n_control):
    pooled_p = (p_treat*n_treat+p_control*n_control) / (n_treat+n_control)
    standard_error = (pooled_p * (1 - pooled_p) * (1/n_treat + 1/n_control)) ** 0.5
    z = (p_treat - p_control) / standard_error
    prob = 1 - norm.cdf(abs(z))
    return 2*prob

def proportion_test_ci(p_treat, p_control, n_treat, n_control, conf=0.95):
    pooled_p = (p_treat*n_treat+p_control*n_control) / (n_treat+n_control)
    standard_error = (pooled_p * (1 - pooled_p) * (1/n_treat + 1/n_control)) ** 0.5
    diff = p_treat-p_control
    z_crit = norm.isf((1-conf)/2)
    return (diff - z_crit*standard_error, diff + z_crit*standard_error)

### Example usage
We ran an experiment to measure the effect of a new hero image on signup rate. For the control group we saw signup rate of 10% across 5500 visits, and for the treatment group we saw signup rate of 10.5% across 5000 visits, can we reject the null hypothesis at the confidence level of 5%? And what's the confidence level around the improvement?

In [63]:
print('p-value of the one-sided test is {:0.1%}'.format(proportion_test_1side(0.105, 0.1, 5000, 5500)))

p-value of the one-sided test is 19.9%


In [64]:
print('p-value of the two-sided test is {:0.1%}'.format(proportion_test_2side(0.105, 0.1, 5000, 5500)))

p-value of the two-sided test is 39.9%


In [66]:
print('The 95% confidence interval of the improvement is between ({:0.1%}, {:0.1%})'.format(*proportion_test_ci(0.105, 0.1, 5000, 5500)))

The 95% confidence interval of the improvement is between (-0.7%, 1.7%)


#### We cannot conclude that treatment is better than control at a confidence level of 5% (19.9% > 5%)